This notebook has the code for how I go sentiment classification results, labse sentence embeddings, stanza postags and lemmas

you need to run this in google colab with a GPU

it has bits and pieces of code from these:
- sentiment-roberta-prediction-example
https://huggingface.co/siebert/sentiment-roberta-large-english
- https://huggingface.co/j-hartmann/emotion-english-distilroberta-base
- https://huggingface.co/sentence-transformers/LaBSE
- stanza and nltk too

In [ ]:
# # Install the transformers library
# !pip install datasets transformers==4.28.0
# !pip install --upgrade accelerate

In [2]:
# Import required packages
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# Create class for data preparation
class SimpleDataset:
    def __init__(self, tokenized_texts):
        self.tokenized_texts = tokenized_texts

    def __len__(self):
        return len(self.tokenized_texts["input_ids"])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.tokenized_texts.items()}

In [3]:
# Load tokenizer and model, create trainer
model_name = "siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
trainer = Trainer(model=model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [4]:
# Create list of texts (can be imported from .csv, .xls etc.)
#pred_texts = ['I like that','That is annoying','This is great!','Wouldn´t recommend it.']

In [10]:
# Example: Import data from csv-file stored on Google Drive

from google.colab import drive
drive.mount('/content/drive')


file_name = "/content/drive/Shareddrives/DHSI 2026/csv/montagu_letters_v2.csv"
text_column = "body"

df_pred = pd.read_csv(file_name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [16]:
# split letter into sentences because it operates on sentences
df_pred["sentences"] = df_pred["body"].apply(lambda x: nltk.sent_tokenize(x))

In [19]:
# \n is a bit weird, clean it
df_pred["sentences"] = df_pred["sentences"].apply(
    lambda sent_list: [s.replace('\n', ' ').strip() for s in sent_list]
)

In [20]:
df_pred["sentences"].iloc[0]

['I FLATTER, myself, dear sister, that I shall give you some pleasure in letting you know that I have safely passed the sea, though we had the ill fortune of a storm.',
 'We were persuaded by the captain of the yacht to set out in a calm, and he pretended there was nothing so easy as to tide it over; but, after two days slowly moving, the wind blew so hard, that none of the sailors could keep their feet, and we were all Sunday night tossed very handsomely.',
 'I never saw a man more frighted (sic) than the captain.',
 'For my part, I have been so lucky, neither to suffer from fear nor seasickness; though, I confess, I was so impatient to see myself once more upon dry land, that I would not stay till the yacht could get to Rotterdam, but went in the long-boat to Helvoetsluys, where we had voitures to carry us to the Briel.',
 'I was charmed with the neatness of that little town; but my arrival at Rotterdam presented me a new scene of pleasure.',
 'All the streets are paved with broad st

In [21]:
# 1. Flatten: keep track of which letter each sentence belongs to
letter_ids = []
pred_texts = []

for i, sent_list in enumerate(df_pred["sentences"]):
    for s in sent_list:
        letter_ids.append(i)
        pred_texts.append(s)

# 2. Tokenize texts and create prediction data set (exactly like the original)
tokenized_texts = tokenizer(pred_texts, truncation=True, padding=True)
pred_dataset = SimpleDataset(tokenized_texts)

# 3. Run predictions (exactly like the original)
predictions = trainer.predict(pred_dataset)

# 4. Transform predictions to labels (exactly like the original)
preds = predictions.predictions.argmax(-1)
labels = pd.Series(preds).map(model.config.id2label)
scores = (np.exp(predictions[0]) / np.exp(predictions[0]).sum(-1, keepdims=True)).max(1)

# 5. Build a flat results dataframe — one row per sentence
df_sentences = pd.DataFrame({
    "letter_idx": letter_ids,
    "sentence": pred_texts,
    "pred": preds,
    "label": labels,
    "score": scores
})

df_sentences.head()

,letter_idx,sentence,pred,label,score
0,0,"I FLATTER, myself, dear sister, that I shall g...",0,NEGATIVE,0.995465
1,0,We were persuaded by the captain of the yacht ...,0,NEGATIVE,0.998392
2,0,I never saw a man more frighted (sic) than the...,0,NEGATIVE,0.992884
3,0,"For my part, I have been so lucky, neither to ...",1,POSITIVE,0.997867
4,0,I was charmed with the neatness of that little...,1,POSITIVE,0.998731


In [22]:
df_sentences.to_csv("/content/drive/Shareddrives/DHSI 2026/csv/montagu_letters_siebert_sentiment.csv", index=False)


https://huggingface.co/j-hartmann/emotion-english-distilroberta-base

In [23]:
from transformers import pipeline

# emotion_classifier = pipeline(
#     "text-classification",
#     model="j-hartmann/emotion-english-distilroberta-base",
#     return_all_scores=True
# )

# # Run on the same flattened sentence list used for siebert
# emotion_results = emotion_classifier(pred_texts, truncation=True, max_length=512)


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [31]:


emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k = None
)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [32]:
emotion_results = emotion_classifier(pred_texts, truncation=True, max_length=512)


In [33]:
emotion_results

[[{'label': 'joy', 'score': 0.967958927154541},
  {'label': 'neutral', 'score': 0.017891759052872658},
  {'label': 'sadness', 'score': 0.006394340191036463},
  {'label': 'surprise', 'score': 0.003316138405352831},
  {'label': 'anger', 'score': 0.001776458928361535},
  {'label': 'fear', 'score': 0.0013899023178964853},
  {'label': 'disgust', 'score': 0.0012724592816084623}],
 [{'label': 'anger', 'score': 0.37339890003204346},
  {'label': 'neutral', 'score': 0.1702508181333542},
  {'label': 'sadness', 'score': 0.16323256492614746},
  {'label': 'disgust', 'score': 0.14975903928279877},
  {'label': 'joy', 'score': 0.05528762564063072},
  {'label': 'surprise', 'score': 0.05496210604906082},
  {'label': 'fear', 'score': 0.033108945935964584}],
 [{'label': 'fear', 'score': 0.9823507070541382},
  {'label': 'neutral', 'score': 0.006064228247851133},
  {'label': 'disgust', 'score': 0.003090426791459322},
  {'label': 'anger', 'score': 0.003022572258487344},
  {'label': 'surprise', 'score': 0.0029

In [27]:
# top_labels = []
# top_scores = []

# for r in emotion_results:
#     top_labels.append(r["label"])
#     top_scores.append(r["score"])

# df_emotions = pd.DataFrame({
#     "letter_idx": letter_ids,
#     "sentence": pred_texts,
#     "top_emotion": top_labels,
#     "top_emotion_score": top_scores
# })

# df_emotions.head()

,letter_idx,sentence,top_emotion,top_emotion_score
0,0,"I FLATTER, myself, dear sister, that I shall g...",joy,0.967959
1,0,We were persuaded by the captain of the yacht ...,anger,0.373399
2,0,I never saw a man more frighted (sic) than the...,fear,0.982351
3,0,"For my part, I have been so lucky, neither to ...",fear,0.777251
4,0,I was charmed with the neatness of that little...,joy,0.990140


In [28]:
# df_emotions.to_csv("/content/drive/Shareddrives/DHSI 2026/csv/montagu_letters_hartmann_emotion_results.csv", index=False)

In [34]:
# emotion_results is now: list (one per sentence) of lists of 7 dicts each
top_labels = []
top_scores = []
all_scores_dicts = []

for scores in emotion_results:
    best = max(scores, key=lambda x: x["score"])
    top_labels.append(best["label"])
    top_scores.append(best["score"])
    all_scores_dicts.append({s["label"]: s["score"] for s in scores})

df_emotions_full = pd.DataFrame({
    "letter_idx": letter_ids,
    "sentence": pred_texts,
    "top_emotion": top_labels,
    "top_emotion_score": top_scores
})

emotion_breakdown_df = pd.DataFrame(all_scores_dicts)
df_emotions_full = pd.concat([df_emotions_full, emotion_breakdown_df], axis=1)

df_emotions_full.head()

,letter_idx,sentence,top_emotion,top_emotion_score,joy,neutral,sadness,surprise,anger,fear,disgust
0,0,"I FLATTER, myself, dear sister, that I shall g...",joy,0.967959,0.967959,0.017892,0.006394,0.003316,0.001776,0.001390,0.001272
1,0,We were persuaded by the captain of the yacht ...,anger,0.373399,0.055288,0.170251,0.163233,0.054962,0.373399,0.033109,0.149759
2,0,I never saw a man more frighted (sic) than the...,fear,0.982351,0.000928,0.006064,0.001604,0.002940,0.003023,0.982351,0.003090
3,0,"For my part, I have been so lucky, neither to ...",fear,0.777251,0.016553,0.010789,0.181294,0.002761,0.008752,0.777251,0.002600
4,0,I was charmed with the neatness of that little...,joy,0.990140,0.990140,0.002601,0.001410,0.000853,0.001425,0.000437,0.003134


In [36]:
emotion_cols = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

def second_highest(row):
    sorted_emotions = sorted(
        [(col, row[col]) for col in emotion_cols],
        key=lambda x: x[1],
        reverse=True
    )
    second_label, second_score = sorted_emotions[1]
    return pd.Series([second_label, second_score])

df_emotions_full[["second_emotion", "second_emotion_score"]] = df_emotions_full.apply(second_highest, axis=1)
df_emotions_full["confidence_gap"] = df_emotions_full["top_emotion_score"] - df_emotions_full["second_emotion_score"]

df_emotions_full.head()

,letter_idx,sentence,top_emotion,top_emotion_score,joy,neutral,sadness,surprise,anger,fear,disgust,second_emotion_score,confidence_gap,second_emotion
0,0,"I FLATTER, myself, dear sister, that I shall g...",joy,0.967959,0.967959,0.017892,0.006394,0.003316,0.001776,0.001390,0.001272,0.017892,0.950067,neutral
1,0,We were persuaded by the captain of the yacht ...,anger,0.373399,0.055288,0.170251,0.163233,0.054962,0.373399,0.033109,0.149759,0.170251,0.203148,neutral
2,0,I never saw a man more frighted (sic) than the...,fear,0.982351,0.000928,0.006064,0.001604,0.002940,0.003023,0.982351,0.003090,0.006064,0.976286,neutral
3,0,"For my part, I have been so lucky, neither to ...",fear,0.777251,0.016553,0.010789,0.181294,0.002761,0.008752,0.777251,0.002600,0.181294,0.595958,sadness
4,0,I was charmed with the neatness of that little...,joy,0.990140,0.990140,0.002601,0.001410,0.000853,0.001425,0.000437,0.003134,0.003134,0.987005,disgust


In [38]:
df_emotions_full.to_csv("/content/drive/Shareddrives/DHSI 2026/csv/montagu_letters_hartmann_emotion_full.csv", index=False)

In [12]:
# Tokenize texts and create prediction data set
# tokenized_texts = tokenizer(pred_texts,truncation=True,padding=True)
# pred_dataset = SimpleDataset(tokenized_texts)

In [6]:
# Run predictions
# predictions = trainer.predict(pred_dataset)

In [7]:
# Transform predictions to labels
# preds = predictions.predictions.argmax(-1)
# labels = pd.Series(preds).map(model.config.id2label)
# scores = (np.exp(predictions[0])/np.exp(predictions[0]).sum(-1,keepdims=True)).max(1)

In [ ]:
# Create DataFrame with texts, predictions, labels, and scores
# df = pd.DataFrame(list(zip(pred_texts,preds,labels,scores)), columns=['text','pred','label','score'])
# df.head()

Let's get the labse embeddings while we are here

In [39]:
# --- Step 0: Install dependencies ---
# !pip install -U sentence-transformers nltk

import pandas as pd
import numpy as np
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

# --- Step 1: Load the combined letters CSV ---
# Update this path to wherever you uploaded the file in Colab
df = pd.read_csv("/content/drive/Shareddrives/DHSI 2026/csv/letters_body.csv")

print("Shape:", df.shape)
print(df.columns.tolist())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Shape: (52, 3)
['letter_id', 'body_en', 'body_fr']


In [40]:
# --- Step 2: Clean + tokenize each language into sentences ---
df["body_en"] = df["body_en"].astype(str)
df["body_fr"] = df["body_fr"].astype(str)

df["sentences_en"] = df["body_en"].apply(
    lambda x: [s.replace("\n", " ").strip() for s in nltk.sent_tokenize(x)]
)
df["sentences_fr"] = df["body_fr"].apply(
    lambda x: [s.replace("\n", " ").strip() for s in nltk.sent_tokenize(x, language="french")]
)

In [41]:
df.head()

,letter_id,body_en,body_fr,sentences_en,sentences_fr
0,LTR-001,"I FLATTER, myself, dear sister, that I shall g...","Vous apprendrez avec plaisir, sans doute, ma c...","[I FLATTER, myself, dear sister, that I shall ...","[Vous apprendrez avec plaisir, sans doute, ma ..."
1,LTR-002,"I MAKE haste to tell you, dear Madam, that, af...","JE me hâte de vous apprendre, ma chere Dame, q...","[I MAKE haste to tell you, dear Madam, that, a...","[JE me hâte de vous apprendre, ma chere Dame, ..."
2,LTR-003,"I AM extremely sorry, my dear S. that your fea...","JE suis bien fâchée, ma chere S. C. que l'inqu...","[I AM extremely sorry, my dear S. that your fe...","[JE suis bien fâchée, ma chere S., C. que l'in..."
3,LTR-004,IF my lady ---- could have any notion of the ...,"SI vous pouviez, Milady, vous former une idée ...",[IF my lady ---- could have any notion of the...,"[SI vous pouviez, Milady, vous former une idée..."
4,LTR-005,"AFTER five days travelling post, I could not s...",LA fatigue que j'ai essuyée pendant cinq jours...,"[AFTER five days travelling post, I could not ...",[LA fatigue que j'ai essuyée pendant cinq jour...


In [42]:
# --- Step 3: Flatten into one row per sentence, keeping letter_id + language ---
def flatten_sentences(df, text_col, lang_label):
    letter_ids = []
    sentences = []
    for letter_id, sent_list in zip(df["letter_id"], df[text_col]):
        for s in sent_list:
            letter_ids.append(letter_id)
            sentences.append(s)
    return pd.DataFrame({
        "letter_id": letter_ids,
        "sentence": sentences,
        "language": lang_label
    })

df_en_flat = flatten_sentences(df, "sentences_en", "en")
df_fr_flat = flatten_sentences(df, "sentences_fr", "fr")

df_all = pd.concat([df_en_flat, df_fr_flat], ignore_index=True)
df_all["sentence_id"] = df_all.index

print("Total sentences:", len(df_all))
print(df_all["language"].value_counts())

Total sentences: 5460
language
fr    2881
en    2579
Name: count, dtype: int64


In [43]:
# --- Step 4: Load LaBSE and embed everything ---
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE")

sentences_to_embed = df_all["sentence"].tolist()

embeddings = model.encode(
    sentences_to_embed,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)  # (num_sentences, 768)


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Embeddings shape: (5460, 768)


In [44]:
output_dir = "/content/drive/Shareddrives/DHSI 2026/csv"

np.save(f"{output_dir}/montagu_labse_embeddings.npy", embeddings)
df_all.to_csv(f"{output_dir}/montagu_labse_metadata.csv", index=False)

print("Saved embeddings:", embeddings.shape)
print("Saved metadata:", df_all.shape)

Saved embeddings: (5460, 768)
Saved metadata: (5460, 4)


In [ ]:
# # --- Step 6: Quick t-SNE sanity check ---
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt

# tsne = TSNE(n_components=2, random_state=42, perplexity=30)
# embeddings_2d = tsne.fit_transform(embeddings)

# df_all["tsne_x"] = embeddings_2d[:, 0]
# df_all["tsne_y"] = embeddings_2d[:, 1]

# df_all.to_csv(f"{output_dir}/montagu_labse_metadata_with_tsne.csv", index=False)

# plt.figure(figsize=(10, 8))
# for lang, color in zip(["en", "fr"], ["steelblue", "indianred"]):
#     subset = df_all[df_all["language"] == lang]
#     plt.scatter(subset["tsne_x"], subset["tsne_y"], label=lang, alpha=0.5, s=15, color=color)

# plt.legend()
# plt.title("t-SNE of LaBSE embeddings: English vs French letters")
# plt.savefig(f"{output_dir}/montagu_labse_tsne.png", dpi=150, bbox_inches="tight")
# plt.show()

let's try to see if we can get a word embedding going

In [46]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 29.9 MB/s eta 0:00:00


In [47]:
import pandas as pd
import numpy as np
import re
import io
from collections import Counter
import stanza
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

output_dir = "/content/drive/Shareddrives/DHSI 2026/csv"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [48]:
# ============================================================
# Step 1: Load letters
# ============================================================
df = pd.read_csv(f"{output_dir}/letters_body.csv")
df["body_en"] = df["body_en"].astype(str)
df["body_fr"] = df["body_fr"].astype(str)

In [50]:

# ============================================================
# Step 2: Lemmatize EVERYTHING with Stanza (no filtering yet)
# ============================================================
stanza.download("en")
stanza.download("fr")

nlp_en = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma", use_gpu=True)
nlp_fr = stanza.Pipeline(lang="fr", processors="tokenize,mwt,pos,lemma", use_gpu=True)

def lemmatize_text(text, nlp):
    """
    Run Stanza on a block of text, return list of (lemma, pos, original_word) tuples.
    Keeps only alphabetic tokens (drops punctuation, numbers).
    """
    text = text.replace("\n", " ")
    doc = nlp(text)
    results = []
    for sent in doc.sentences:
        for word in sent.words:
            lemma = word.lemma if word.lemma else word.text
            lemma = lemma.lower().strip()
            if lemma.isalpha():  # drop punctuation/numbers
                results.append({
                    "lemma": lemma,
                    "pos": word.upos,       # universal POS tag, e.g. NOUN, VERB, ADJ
                    "xpos": word.xpos,      # language-specific POS tag (more granular)
                    "original": word.text.lower()
                })
    return results

print("Lemmatizing English letters...")
df["tokens_en"] = df["body_en"].apply(lambda t: lemmatize_text(t, nlp_en))

print("Lemmatizing French letters...")
df["tokens_fr"] = df["body_fr"].apply(lambda t: lemmatize_text(t, nlp_fr))

# Save the full tokenized version — one row per (letter, token), with lemma + POS + original word.
# This is much more useful for class than a flat list of lemmas, since you can now
# filter/group by part of speech later (e.g. "show me only nouns" or "only verbs").
def explode_tokens(df, token_col, lang_label):
    rows = []
    for letter_id, tokens in zip(df["letter_id"], df[token_col]):
        for t in tokens:
            rows.append({
                "letter_id": letter_id,
                "language": lang_label,
                "lemma": t["lemma"],
                "pos": t["pos"],
                "xpos": t["xpos"],
                "original_word": t["original"],
            })
    return pd.DataFrame(rows)

df_tokens_en = explode_tokens(df, "tokens_en", "en")
df_tokens_fr = explode_tokens(df, "tokens_fr", "fr")
df_tokens_all = pd.concat([df_tokens_en, df_tokens_fr], ignore_index=True)

df_tokens_all.to_csv(f"{output_dir}/montagu_lemmatized_pos.csv", index=False)
print("Saved full lemma+POS table:", df_tokens_all.shape)
df_tokens_all.head()



INFO:stanza:Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/.cache/stanza/1.13.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.13.0/resources


INFO:stanza:Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
INFO:stanza:Downloading default packages for language: fr (French) ...
INFO:stanza:File exists: /root/.cache/stanza/1.13.0/resources/fr/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.13.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
INFO:stanza:Loading these models for language: fr (French):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


Lemmatizing English letters...
Lemmatizing French letters...
Saved full lemma+POS table: (120624, 6)


,letter_id,language,lemma,pos,xpos,original_word
0,LTR-001,en,i,PRON,PRP,i
1,LTR-001,en,flatter,ADV,RB,flatter
2,LTR-001,en,myself,PRON,PRP,myself
3,LTR-001,en,dear,ADJ,JJ,dear
4,LTR-001,en,sister,NOUN,NN,sister


In [51]:

# ============================================================
# Step 3: Build full vocabulary (lemma counts), NO filtering yet
# ============================================================
en_lemma_counts = Counter(df_tokens_en["lemma"])
fr_lemma_counts = Counter(df_tokens_fr["lemma"])

print("Unique EN lemmas (unfiltered):", len(en_lemma_counts))
print("Unique FR lemmas (unfiltered):", len(fr_lemma_counts))

# We keep the FULL vocab for embedding lookup — no min-count or stopword
# filtering at this stage, per your point: embed everything, decide what
# to exclude from the PLOT afterward.
en_vocab_full = set(en_lemma_counts.keys())
fr_vocab_full = set(fr_lemma_counts.keys())

# Also build a lemma -> most common POS lookup, useful later for filtering
# by part of speech (e.g. plot only nouns/verbs) or annotating the word plot
en_lemma_to_pos = (
    df_tokens_en.groupby("lemma")["pos"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)
fr_lemma_to_pos = (
    df_tokens_fr.groupby("lemma")["pos"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)


Unique EN lemmas (unfiltered): 4868
Unique FR lemmas (unfiltered): 5082


In [52]:
# ============================================================
# Step 4: Download aligned fastText vectors, filtered to our full vocab
# ============================================================
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-aligned/wiki.en.align.vec -O wiki.en.align.vec
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-aligned/wiki.fr.align.vec -O wiki.fr.align.vec

def load_filtered_vectors(filepath, vocab):
    vectors = {}
    with io.open(filepath, "r", encoding="utf-8", newline="\n", errors="ignore") as f:
        n, dim = map(int, f.readline().split())
        for line in f:
            tokens = line.rstrip().split(" ")
            word = tokens[0]
            if word in vocab:
                vectors[word] = np.array(tokens[1:], dtype=np.float32)
    return vectors

print("Loading EN vectors (filtered to lemmatized vocab)...")
en_vectors = load_filtered_vectors("wiki.en.align.vec", en_vocab_full)
print("Matched EN lemmas:", len(en_vectors), "/", len(en_vocab_full))

print("Loading FR vectors (filtered to lemmatized vocab)...")
fr_vectors = load_filtered_vectors("wiki.fr.align.vec", fr_vocab_full)
print("Matched FR lemmas:", len(fr_vectors), "/", len(fr_vocab_full))

Loading EN vectors (filtered to lemmatized vocab)...
Matched EN lemmas: 4619 / 4868
Loading FR vectors (filtered to lemmatized vocab)...
Matched FR lemmas: 4421 / 5082


In [53]:
# ============================================================
# Step 5: Build combined dataframe — ALL matched words, with frequency
# ============================================================
en_rows = [{"word": w, "language": "en", "count": en_lemma_counts[w],
            "pos": en_lemma_to_pos.get(w, "X"), "vector": v}
           for w, v in en_vectors.items()]
fr_rows = [{"word": w, "language": "fr", "count": fr_lemma_counts[w],
            "pos": fr_lemma_to_pos.get(w, "X"), "vector": v}
           for w, v in fr_vectors.items()]

df_words = pd.DataFrame(en_rows + fr_rows)
print("Total lemmas with vectors (before stopword filtering):", len(df_words))

word_vectors = np.stack(df_words["vector"].values)
print("word_vectors shape:", word_vectors.shape)

# Save the FULL (unfiltered) embeddings — your master file, reusable
# for any future filtering experiment without re-downloading fastText
np.save(f"{output_dir}/montagu_fasttext_lemma_vectors_FULL.npy", word_vectors)
df_words.drop(columns=["vector"]).to_csv(f"{output_dir}/montagu_fasttext_lemma_metadata_FULL.csv", index=False)

Total lemmas with vectors (before stopword filtering): 9040
word_vectors shape: (9040, 300)


In [54]:
# ============================================================
# Step 6: Filter to NOUNS only — cleanest single view for class
# ============================================================
df_words_filtered = df_words[df_words["pos"] == "NOUN"].reset_index(drop=True)

print("Words remaining after filtering to NOUN only:", len(df_words_filtered))
print(df_words_filtered["language"].value_counts())

word_vectors_filtered = np.stack(df_words_filtered["vector"].values)
print("Filtered word_vectors shape:", word_vectors_filtered.shape)

Words remaining after filtering to NOUN only: 4111
language
en    2083
fr    2028
Name: count, dtype: int64
Filtered word_vectors shape: (4111, 300)


In [ ]:
# # ============================================================
# # Step 6: NOW filter stopwords — combine NLTK + frequency-based + manual
# # ============================================================
# en_stopwords = set(stopwords.words("english"))
# fr_stopwords = set(stopwords.words("french"))

# # Frequency-based: drop top N most frequent lemmas in YOUR corpus.
# # This is corpus-adaptive, which you found worked better than a fixed list alone.
# TOP_N_FREQ_CUTOFF = 100

# top_en_by_freq = {w for w, _ in en_lemma_counts.most_common(TOP_N_FREQ_CUTOFF)}
# top_fr_by_freq = {w for w, _ in fr_lemma_counts.most_common(TOP_N_FREQ_CUTOFF)}

# # Manual additions for things neither list catches (editorial markers, etc.)
# extra_en_stopwords = {"sic", "c"}
# extra_fr_stopwords = {"c", "j", "n", "s", "l", "d", "m", "t", "y", "qu"}

# en_exclude = en_stopwords | top_en_by_freq | extra_en_stopwords
# fr_exclude = fr_stopwords | top_fr_by_freq | extra_fr_stopwords

# df_words_filtered = df_words[
#     ~((df_words["language"] == "en") & (df_words["word"].isin(en_exclude))) &
#     ~((df_words["language"] == "fr") & (df_words["word"].isin(fr_exclude)))
# ].reset_index(drop=True)

# print("Words remaining after stopword + top-N frequency filtering:", len(df_words_filtered))
# print(df_words_filtered["language"].value_counts())

# word_vectors_filtered = np.stack(df_words_filtered["vector"].values)
# print("Filtered word_vectors shape:", word_vectors_filtered.shape)

In [55]:
# ============================================================
# Step 7: Save the FILTERED version (what you'll actually plot)
# ============================================================
np.save(f"{output_dir}/montagu_fasttext_lemma_vectors_filtered.npy", word_vectors_filtered)
df_words_filtered.drop(columns=["vector"]).to_csv(
    f"{output_dir}/montagu_fasttext_lemma_metadata_filtered.csv", index=False
)

In [56]:

# ============================================================
# Step 8: t-SNE + interactive plot on the FILTERED set
# ============================================================
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

n_words = word_vectors_filtered.shape[0]
perplexity = min(30, max(5, n_words // 10))

tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
word_embeddings_2d = tsne.fit_transform(word_vectors_filtered)

df_words_filtered["tsne_x"] = word_embeddings_2d[:, 0]
df_words_filtered["tsne_y"] = word_embeddings_2d[:, 1]

df_words_filtered.drop(columns=["vector"]).to_csv(
    f"{output_dir}/montagu_fasttext_lemma_metadata_filtered_with_tsne.csv", index=False
)

fig = px.scatter(
    df_words_filtered,
    x="tsne_x", y="tsne_y",
    color="language",
    size="count",
    size_max=18,
    color_discrete_map={"en": "#4C78A8", "fr": "#E45756"},
    custom_data=["word", "count", "language", "pos"],
    title="FastText Aligned Word Embeddings (Lemmatized): English vs French",
    opacity=0.7,
    width=950, height=700,
)
fig.update_traces(
    hovertemplate="<b>%{customdata[0]}</b> (%{customdata[3]})<br>language: %{customdata[2]}<br>count: %{customdata[1]}<extra></extra>"
)
fig.update_layout(
    legend_title_text="Language",
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#eee", zeroline=False, title="t-SNE dim 1"),
    yaxis=dict(showgrid=True, gridcolor="#eee", zeroline=False, title="t-SNE dim 2"),
)
fig.show()
fig.write_html(f"{output_dir}/montagu_fasttext_lemma_tsne_interactive.html")
print("Saved interactive HTML.")



Saved interactive HTML.


In [ ]:

# # ============================================================
# # OPTIONAL: word search/highlight (same pattern as before)
# # ============================================================
# def plot_word_with_highlight(df, keyword_words, case_sensitive=False):
#     if not case_sensitive:
#         keyword_words_lower = [w.lower() for w in keyword_words]
#         mask = df["word"].str.lower().isin(keyword_words_lower)
#     else:
#         mask = df["word"].isin(keyword_words)

#     df_match = df[mask]
#     df_rest = df[~mask]

#     fig = go.Figure()
#     fig.add_trace(go.Scatter(
#         x=df_rest["tsne_x"], y=df_rest["tsne_y"],
#         mode="markers",
#         marker=dict(size=6, color="#dddddd", opacity=0.4),
#         customdata=df_rest[["word", "language"]],
#         hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<extra></extra>",
#         name="other words", showlegend=False,
#     ))
#     for lang, color in zip(["en", "fr"], ["#4C78A8", "#E45756"]):
#         sub = df_match[df_match["language"] == lang]
#         fig.add_trace(go.Scatter(
#             x=sub["tsne_x"], y=sub["tsne_y"],
#             mode="markers+text",
#             text=sub["word"],
#             textposition="top center",
#             marker=dict(size=12, color=color, line=dict(width=1.5, color="black")),
#             customdata=sub[["word", "language"]],
#             hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<extra></extra>",
#             name=f"{lang} (match)",
#         ))
#     fig.update_layout(
#         title=f"Highlighted lemmas: {', '.join(keyword_words)}",
#         plot_bgcolor="white",
#         xaxis=dict(showgrid=True, gridcolor="#eee", zeroline=False, title="t-SNE dim 1"),
#         yaxis=dict(showgrid=True, gridcolor="#eee", zeroline=False, title="t-SNE dim 2"),
#         width=950, height=700,
#     )
#     fig.show()
#     return df_match

# # Example: plot_word_with_highlight(df_words_filtered, ["plaisir", "pleasure", "ville", "town"])